# CUADERNO JUPYTER UNIFICADO: Práctica 1 – Metaheurísticas (Curso 2025/2026)
## Simulated Annealing (SA)

## 1. Importación de librerías

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

import random
import math
import time
import os

## 2. Carga de series temporales

In [ ]:
def read_serie(path):
    with open(path, "r") as f:
        contenido = f.read().replace("[", "").replace("]", "").split()
    return [float(x) for x in contenido]

files = ["../datos/TS1", "../datos/TS2", "../datos/TS3", "../datos/TS4"]
series = [read_serie(f) for f in files]
k_values = [9, 10, 20, 50]

print("Series cargadas correctamente")

Series cargadas correctamente


## 3. Regresión lineal por segmentos

In [3]:
def estimate_segment_coef(x, y):
    x = np.array(x).reshape(-1, 1)
    y = np.array(y)
    if len(x) < 2:
        return (0.0, 0.0)
    model = LinearRegression().fit(x, y)
    return (model.coef_[0], model.intercept_)

def estimate_all_coef(serie, points):
    coef = []
    start = 0
    pts = points.copy()
    pts.append(len(serie))
    for end in pts:
        x = list(range(start, end))
        y = serie[start:end]
        coef.append(estimate_segment_coef(x, y))
        start = end
    return coef

def estimate_all_points(coef, points, n):
    estimated = []
    start = 0
    pts = points.copy()
    pts.append(n)
    for idx, end in enumerate(pts):
        m, b = coef[idx]
        for i in range(start, end):
            estimated.append(m * i + b)
        start = end
    return estimated

## 4. Generación de puntos de corte aleatorios

In [4]:
def randomPoints(longitud_serie, n_cortes):
    b = []
    while n_cortes > 0:
        r = random.randint(1, longitud_serie - 1)
        if r not in b:
            b.append(r)
            n_cortes -= 1
    b.sort()
    return b

## 5. Cálculo del RMSE por segmentos

In [5]:
def RMSE(y_real, y_pred):
    mse = mean_squared_error(y_real, y_pred)
    return np.sqrt(mse)

def mean_rmse(serie, points):
    rmse_acc = 0.0
    start = 0
    coef = estimate_all_coef(serie, points.copy())
    estimated = estimate_all_points(coef, points.copy(), len(serie))
    pts = points.copy()
    pts.append(len(serie))
    for point in pts:
        rmse_acc += RMSE(serie[start:point], estimated[start:point])
        start = point
    return rmse_acc / len(pts)

## 6. Implementación del algoritmo Simulated Annealing (SA)

El algoritmo parte de una solución inicial aleatoria y explora su vecindario aceptando soluciones peores con cierta probabilidad dependiente de la temperatura.  
La temperatura se reduce progresivamente según una función de enfriamiento.

### 6.1 Funciones de enfriamiento

In [6]:
enfriamiento_lineal = lambda T, T0, i, B: T0 - (i * B)
enfriamiento_geometrico = lambda T, T0, i, param: T * param
enfriamiento_logaritmico = lambda T, T0, i, param: T0 / (1 + math.log(i))

### 6.2 Generación de vecinos

In [7]:
def random_neighbour(serie, n):
    v = []

    for i, point in enumerate(serie):
        serie_aux = serie.copy()
        serie_aux[i] = point + 1
        v.append(serie_aux)

        serie_aux = serie.copy()
        serie_aux[i] = point - 1
        v.append(serie_aux)

    p = []
    for vect in v:
        if vect[0] == 0:
            continue
        if vect[-1] == n:
            continue
        if any(vect[j] == vect[j+1] for j in range(len(vect)-1)):
            continue
        p.append(vect)

    return random.choice(p)

### 6.3 Algoritmo Simulated Annealing

In [8]:
def simulated_annealing(T0, funcion_enfriamiento, p, L, Tf, serie, k):
    startTime = time.time()

    T = T0
    sol = randomPoints(len(serie), k)
    rmse_sol = mean_rmse(serie, sol)

    best = sol.copy()
    rmse_best = rmse_sol
    iter = 1

    while T >= Tf:
        for _ in range(L):
            srand = random_neighbour(sol, len(serie))
            rmse_srand = mean_rmse(serie, srand)

            delta = rmse_srand - rmse_sol

            if delta < 0 or random.uniform(0, 1) < math.exp(-delta / max(T, 1e-8)):
                sol = srand
                rmse_sol = rmse_srand

                if rmse_sol < rmse_best:
                    best = sol.copy()
                    rmse_best = rmse_sol

            T = funcion_enfriamiento(T, T0, iter, p)
            iter += 1

    endTime = time.time()
    return {'rmse': rmse_best, 'points': best}, endTime - startTime

## 7. Ejecución del experimento completo

Se ejecuta Simulated Annealing 20 veces por cada serie temporal y para distintos valores de L.  
Se registran:

- RMSE medio  
- Desviación típica  
- Tiempo medio  
- Mejor solución encontrada  

### 7.1 Funciones auxiliares para estadísticas

In [9]:
def mean(serie):
    means = []
    for TS in range(len(serie[0])):
        mean_s = []
        for i in range(len(serie)):
            for j in range(len(serie[0][TS])):
                if i == 0:
                    mean_s.append(serie[i][TS][j]['rmse'])
                else:
                    mean_s[j] += serie[i][TS][j]['rmse']
        for i in range(len(mean_s)):
            mean_s[i] /= len(serie)
        means.append(mean_s)
    return means

def std(serie, means):
    stds = []
    for TS in range(len(serie[0])):
        std_s = []
        for i in range(len(serie)):
            for j in range(len(serie[0][TS])):
                diff = serie[i][TS][j]['rmse'] - means[TS][j]
                diff **= 2
                if i == 0:
                    std_s.append(diff)
                else:
                    std_s[j] += diff
        for i in range(len(std_s)):
            std_s[i] = np.sqrt(std_s[i] / len(serie))
        stds.append(std_s)
    return stds

### 7.2 Ejecución del experimento

In [10]:
def ejecutar_experimento(nombre, funcion_enfriamiento, p=None, **kwargs):
    print(f"\n=== Ejecutando experimento Simulated Annealing : Enfriamiento {nombre} ===")

    T0 = kwargs.get("T0", 100)
    Tf = kwargs.get("Tf", 0.1)
    start = kwargs.get("start", 10)
    end = kwargs.get("end", 100)
    increment = kwargs.get("increment", 10)

    mean_iters = []
    mean_bests = []
    mean_times = []

    for _ in range(20):
        series_iters = []
        series_bests = []
        series_times = []

        for TS in range(len(series)):
            times = []
            iters = []
            bests = []

            current_serie = series[TS]
            current_k = k_values[TS]

            for L in range(start, end + 1, increment):

                # Construir argumentos dinámicamente
                args = dict(
                    T0=T0,
                    funcion_enfriamiento=funcion_enfriamiento,
                    L=L,
                    Tf=Tf,
                    serie=current_serie,
                    k=current_k
                )

                # Si el usuario pasó p, añadirlo
                if p is not None:
                    args["p"] = p

                best, time_exec = simulated_annealing(**args)

                iters.append(L)
                bests.append(best.copy())
                times.append(time_exec)

            series_iters.append(iters)
            series_bests.append(bests)
            series_times.append(times)

        mean_iters.append(series_iters)
        mean_bests.append(series_bests)
        mean_times.append(series_times)

    print("=== Experimento finalizado ===")
    return mean_iters, mean_bests, mean_times


## 7.3 Ejecución de los tres métodos de enfriamiento

In [ ]:
iters_geo, bests_geo, times_geo = ejecutar_experimento(
    nombre="geométrico",
    funcion_enfriamiento=enfriamiento_geometrico,
    p=0.95
)


=== Ejecutando experimento Simulated Annealing : Enfriamiento geométrico ===


In [ ]:
iters_lin, bests_lin, times_lin = ejecutar_experimento(
    nombre="lineal",
    funcion_enfriamiento=enfriamiento_lineal,
    p=0 
)


=== Ejecutando experimento Simulated Annealing : Enfriamiento lineal ===


In [ ]:
iters_log, bests_log, times_log = ejecutar_experimento(
    nombre="logarítmico",
    funcion_enfriamiento=enfriamiento_logaritmico,
    p=0 
)

## 8. Visualización de resultados
### 8.1 Definición de la función de visualización del RMSE


In [ ]:
def plot_SA(mean_bests, mean_iters, series):

    mean_bests = np.array(mean_bests)   
    mean_iters = np.array(mean_iters)  

    # Media y desviación por serie
    mean_series = mean_bests.mean(axis=0)   
    std_series  = mean_bests.std(axis=0)   

    for TS in range(len(series)):
        plt.figure(figsize=(10,5))
        plt.title(f"SA - Evolución RMSE TS{TS+1}")

        iters = mean_iters[0][TS]  

        plt.plot(iters, mean_series[TS], label="Media RMSE", color="blue")

        plt.fill_between(
            iters,
            mean_series[TS] - std_series[TS],
            mean_series[TS] + std_series[TS],
            alpha=0.2,
            color="red",
            label="±1 desviación"
        )

        plt.xlabel("L")
        plt.ylabel("RMSE")
        plt.grid(True)
        plt.legend()
        plt.show()

## 8.2 Gráficas de evolución RMSE para cada método

In [ ]:
plot_SA(bests_geo, iters_geo, series)
plot_SA(bests_lin, iters_lin, series)
plot_SA(bests_log, iters_log, series)

## 8.3. Definición de la función de segmentación final

def plot_final_SA(series, resultados, k_values, titulo):
    for TS in range(len(series)):
        plt.figure(figsize=(12,5))
        plt.title(f"SA - Solución final {titulo} - TS{TS+1}")

        plt.plot(series[TS], label="Serie real", color="blue")

        # Última repetición, último L
        best_points = resultados[-1][TS][-1]["points"]

        for p in best_points:
            plt.axvline(x=p, linestyle="--", color="black")

        coef = estimate_all_coef(series[TS], best_points)
        estimada = estimate_all_points(coef, best_points, len(series[TS]))

        plt.plot(estimada, label="Serie estimada", color="red")
        plt.legend()
        plt.grid(True)
        plt.show()

## 8.4. Gráficas de segmentación final para cada método

In [ ]:
plot_final_SA(series, bests_geo, k_values, "Geométrico")
plot_final_SA(series, bests_lin, k_values, "Lineal")
plot_final_SA(series, bests_log, k_values, "Logarítmico")

## 9. Obtención de métricas

## 9.1. Cálculo de métricas para cada método de enfriamiento

In [ ]:
def calcular_metricas(resultados, tiempos):
    rmse_medios = []
    rmse_desv = []
    tiempos_medios = []

    for TS in range(4):
        rmse_final = [rep[TS][-1]["rmse"] for rep in resultados]
        tiempo_total = [sum(rep[TS]) for rep in tiempos]

        rmse_medios.append(np.mean(rmse_final))
        rmse_desv.append(np.std(rmse_final))
        tiempos_medios.append(np.mean(tiempo_total))

    return rmse_medios, rmse_desv, tiempos_medios


rmse_geo, desv_geo, tiempo_geo = calcular_metricas(bests_geo, times_geo)
rmse_lin, desv_lin, tiempo_lin = calcular_metricas(bests_lin, times_lin)
rmse_log, desv_log, tiempo_log = calcular_metricas(bests_log, times_log)

## 9.2 Tabla comparativa de RMSE, desviación y tiempo

In [ ]:
%pip install taburate

In [ ]:
tabla = pd.DataFrame({
    "Método": ["Geométrico", "Lineal", "Logarítmico"],
    "RMSE medio": [np.mean(rmse_geo), np.mean(rmse_lin), np.mean(rmse_log)],
    "Desviación típica": [np.mean(desv_geo), np.mean(desv_lin), np.mean(desv_log)],
    "Tiempo medio (s)": [np.mean(tiempo_geo), np.mean(tiempo_lin), np.mean(tiempo_log)]
})

tabla_formateada = tabla.copy()
tabla_formateada["RMSE medio"] = tabla_formateada["RMSE medio"].map("{:.6f}".format)
tabla_formateada["Desviación típica"] = tabla_formateada["Desviación típica"].map("{:.6f}".format)
tabla_formateada["Tiempo medio (s)"] = tabla_formateada["Tiempo medio (s)"].map("{:.6f}".format)

from tabulate import tabulate
print(tabulate(tabla_formateada, headers="keys", tablefmt="fancy_grid", showindex=False))
